In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from features.category_probability_extractor import CategoryProbabilityExtractor
from features.brand_imputer import BrandImputer
from features.pack_size_probability_extractor import PackSizeProbabilityExtractor
from features.data_preprocessor import DataPreprocessor
from features.cross_validation_ensemble import CrossValidationEnsemble
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

category probability

In [3]:
base_lr = LogisticRegression(max_iter=300, random_state=42)
extractor = CategoryProbabilityExtractor(base_model=base_lr, n_splits=5)
extractor.fit(train)

              precision    recall  f1-score   support

       audio       0.90      0.96      0.93      2155
      beauty       1.00      0.97      0.98      1364
       books       0.92      0.99      0.95      1462
   computers       0.97      0.97      0.97      2318
      garden       0.97      0.96      0.97      1050
     kitchen       0.95      0.95      0.95      1864
       music       0.94      0.55      0.69       466
        pets       1.00      0.97      0.99       714
       sport       0.94      0.93      0.94      1580

    accuracy                           0.95     12973
   macro avg       0.95      0.92      0.93     12973
weighted avg       0.95      0.95      0.95     12973



In [4]:
train = extractor.transform(train)
train['declared_category'] = train['declared_category'].fillna('missing')

test = extractor.transform(test)
test['declared_category'] = test['declared_category'].fillna('missing')

brands

In [5]:
bi = BrandImputer(['Noro', 'Velto', 'Aster', 'Mira', 'Tenso'])
train = bi.transform(train)
test = bi.transform(test)

In [6]:
model = LogisticRegression(C=10.0, max_iter=1000)
ppe = PackSizeProbabilityExtractor(model)
ppe.fit(train)
train = ppe.transform(train)
test = ppe.transform(test)

              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00      9329
         2.0       1.00      1.00      1.00      2038
         3.0       1.00      1.00      1.00      1000
         5.0       1.00      1.00      1.00       515

    accuracy                           1.00     12882
   macro avg       1.00      1.00      1.00     12882
weighted avg       1.00      1.00      1.00     12882



In [7]:
train

,id,seller_id,title,description,declared_category,brand,condition,price_pln,shipping_weight_kg,pack_size,target,declared_cat_prob,predicted_category,declared_pack_size_prob,predicted_pack_size
0,OFR_191714271,SEL_7523257,recording mic support — pack of one — Mira A513,Pack of one main items. Microphone stand by Mi...,audio,Mira,used,70.43,5.233,1.0,1,0.812076,audio,0.998536,1.0
1,OFR_517640949,SEL_2053194,in-ear cable headphones — quantity 2 — Noro E315,Included main items: 2. Wired earbuds by Noro....,computers,Noro,used,28.95,0.119,2.0,1,0.008879,audio,0.974337,2.0
2,OFR_634805877,SEL_8620752,Tenso digital food balance / standard / quanti...,Included main items: 1. Kitchen scale by Tenso...,kitchen,Tenso,new,62.05,0.412,1.0,0,0.984746,kitchen,0.986079,1.0
3,OFR_643990737,SEL_8396386,litter scoop — five main items — Velo ...,Cat tray cleaning shovel by Velto. Variant: st...,pets,Velto,used,102.26,0.499,5.0,0,0.922348,pets,0.952642,5.0
4,OFR_260742748,SEL_5029882,LAPTOP SLEEVE — PACK OF FIVE — ASTER F219,Variant: standard. Also includes 3 pcs of free...,computers,Aster,used,144.84,0.856,5.0,0,0.929330,computers,0.985897,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,OFR_938431337,SEL_99230,Aster planting hand protection / standard / tw...,Gardening gloves by Aster. Variant: standard. ...,garden,Aster,new,35.23,0.230,2.0,0,0.972309,garden,0.993862,2.0
15996,OFR_920123932,SEL_4497749,soap bar — 5 pcs — Tenso D800,NaN,beauty,Tenso,new,67.23,1.918,5.0,1,0.549050,beauty,0.921006,5.0
15997,OFR_704033898,SEL_2730346,Aster swimming goggles / standard / 3-pack,Pool eye protection by Aster. Variant: standar...,sport,Aster,new,202.82,NaN,3.0,0,0.968398,sport,0.982638,3.0
15998,OFR_442461332,SEL_7926600,graphics tablet — quantity 1 — Noro D130,Variant: large. Also includes 1 pcs of free ac...,computers,Noro,used,1080.99,0.701,1.0,1,0.964732,computers,0.992730,1.0


In [8]:
from catboost import CatBoostClassifier

CATEGORICAL_FEATURES = ['declared_category', 'predicted_category', 'brand', 'condition']
NUMERICAL_FEATURES = [
    'declared_cat_prob', 'declared_pack_size_prob',
    'cat_prob_delta', 'pack_prob_delta', 'category_mismatch', 'pack_size_mismatch',
    'z_score_price', 'z_score_price_abs', 'z_score_weight', 'z_score_weight_abs',
    'log_unit_price', 'log_unit_weight', 'log_price', 'log_weight',
    'pack_size', 'predicted_pack_size',
    'title_len', 'desc_len', 'has_missing_price', 'has_missing_weight', 'price_per_weight'
]
TEXT_FEATURES = ['title', 'description']
FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES + TEXT_FEATURES

dp = DataPreprocessor()
base_model_cb = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    cat_features=CATEGORICAL_FEATURES,
    text_features=TEXT_FEATURES,
    auto_class_weights='Balanced',
    random_state=42,
)
cve = CrossValidationEnsemble(
    base_model=base_model_cb,
    preprocessor=dp,
    features=FEATURES,
    n_splits=5
)

In [9]:
cve.fit(train)

--- Fold 1 ---
0:	learn: 0.6770003	total: 104ms	remaining: 2m 35s
1:	learn: 0.6632193	total: 171ms	remaining: 2m 7s
2:	learn: 0.6484996	total: 235ms	remaining: 1m 57s
3:	learn: 0.6364399	total: 280ms	remaining: 1m 44s
4:	learn: 0.6226179	total: 334ms	remaining: 1m 39s
5:	learn: 0.6129034	total: 391ms	remaining: 1m 37s
6:	learn: 0.6027296	total: 454ms	remaining: 1m 36s
7:	learn: 0.5962241	total: 502ms	remaining: 1m 33s
8:	learn: 0.5803548	total: 557ms	remaining: 1m 32s
9:	learn: 0.5728084	total: 615ms	remaining: 1m 31s
10:	learn: 0.5664545	total: 681ms	remaining: 1m 32s
11:	learn: 0.5606901	total: 741ms	remaining: 1m 31s
12:	learn: 0.5559066	total: 796ms	remaining: 1m 31s
13:	learn: 0.5496895	total: 852ms	remaining: 1m 30s
14:	learn: 0.5441950	total: 916ms	remaining: 1m 30s
15:	learn: 0.5407955	total: 965ms	remaining: 1m 29s
16:	learn: 0.5367224	total: 1.01s	remaining: 1m 28s
17:	learn: 0.5327157	total: 1.07s	remaining: 1m 28s
18:	learn: 0.5284545	total: 1.14s	remaining: 1m 28s
19:	lear

In [12]:
test_preds = cve.predict_proba(test)

In [14]:
submission = test[['id']].copy()
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)